# Laboratorio 2 - Complejidad y búsqueda de hiperparámetros
 
**Integrantes:**  Pedro Pablo Sanín Trujillo (202221527)


## 1. Contexto, objetivos y herramientas

### 1.1 Contexto: AlpesPlanck

Se estimará `temp_max_manana` y se estudiará cómo la complejidad y la regularización afectan la generalización.

### 1.2 Objetivos

- Comparar regresión polinomial, lineal, Ridge y Lasso.
- Buscar hiperparámetros mediante validación cruzada.
- Analizar el trade-off sesgo-varianza con curvas de validación.
- Seleccionar un modelo considerando desempeño, complejidad y estabilidad.
- Construir intervalos de confianza del 95% mediante bootstrap.

### 1.3 Herramientas

Python, Pandas, NumPy, Scikit-Learn, Matplotlib y Seaborn.

## 2. Preprocesamiento de los datos

Se parte de los mismos archivos crudos de `data/` (uno de entrenamiento y uno de prueba externa). Antes de modelar es necesario limpiar y estandarizar los datos meteorológicos. A continuación se resumen las decisiones clave de esa limpieza; todas se implementan en una única función (`limpiar_datos`) para garantizar que se apliquen exactamente las mismas reglas, de forma determinista, tanto al conjunto de entrenamiento como al de prueba, sin aprender ningún parámetro a partir de los datos:

- **Normalización de categorías**: los textos de `mes`, `sector_viento` y `estacion_anio` llegan en español/inglés, con abreviaturas y errores de digitación; se homogenizan a un único vocabulario mediante diccionarios de equivalencias fijos.
- **Consistencia de fecha y estacionalidad**: cuando `fecha` es nula pero existen `anio` y `dia_del_anio`, se reconstruye la fecha; luego se recalculan `anio`, `dia_del_anio`, `mes` y `estacion_anio` a partir de la fecha ya consolidada, para que estas columnas nunca queden contradictorias entre sí.
- **Corrección de errores cuantitativos conocidos**: se corrigen valores con un dígito extra en `presion_media`, `viento_desv` y `rafaga_desv`; se descartan mediciones de viento imposibles (`viento_norte < -100`); se intercambian pares máximo/mínimo cuando el máximo reportado es menor que el mínimo; se acotan las humedades al rango físico `[0, 100]`; y se corrigen signos/valores inválidos en `rafaga_desv`, `rafaga_media` y `rafaga_min`. Todas estas reglas provienen de límites de negocio conocidos de antemano (no se calculan a partir de la distribución del conjunto de entrenamiento), por lo que pueden aplicarse igual sobre entrenamiento y prueba sin generar fuga de información.
- **Eliminación de duplicados**: se remueven registros exactamente duplicados tras la limpieza anterior.

Como estas reglas son fijas (no se ajustan con los datos), aplicarlas por separado sobre el conjunto de entrenamiento y sobre el conjunto de prueba no introduce data leakage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    cross_validate,
    validation_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    PolynomialFeatures,
    RobustScaler,
    StandardScaler,
)

RANDOM_STATE = 42
DATA_DIR = "./data"
TARGET = "temp_max_manana"

In [ ]:
def obtener_fecha(anio, dia_del_anio):
    if pd.isna(anio) or pd.isna(dia_del_anio):
        return pd.NaT
    return pd.Timestamp(int(anio), 1, 1) + pd.Timedelta(days=int(dia_del_anio) - 1)


def obtener_sector_compass(grados):
    if pd.isna(grados):
        return np.nan
    if not 0 <= grados < 360:
        return np.nan
    sectores = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    return sectores[int((grados + 22.5) // 45) % 8]


def limpiar_datos(df):
    """Replica las reglas de limpieza del Laboratorio 1 sin modificar el DataFrame original."""
    data = df.copy()

    # Normalización de etiquetas categóricas.
    meses = {
        "enero": "january", "febrero": "february", "marzo": "march",
        "abril": "april", "mayo": "may", "junio": "june", "julio": "july",
        "agosto": "august", "septiembre": "september", "octubre": "october",
        "noviembre": "november", "diciembre": "december", "jan": "january",
        "feb": "february", "mar": "march", "apr": "april", "jun": "june",
        "jul": "july", "aug": "august", "sept": "september", "sep": "september",
        "oct": "october", "nov": "november", "dec": "december",
    }
    if "mes" in data:
        data["mes"] = data["mes"].astype("string").str.lower().str.strip().replace(meses).str.capitalize()

    sectores = {
        "norte": "n", "sur": "s", "este": "e", "oeste": "w", "noreste": "ne",
        "sureste": "se", "suroeste": "sw", "noroeste": "nw", "north": "n",
        "south": "s", "east": "e", "west": "w", "northeast": "ne",
        "southeast": "se", "northwest": "nw", "southwest": "sw", "no": "nw",
        "o": "w", "so": "sw",
    }
    if "sector_viento" in data:
        data["sector_viento"] = (
            data["sector_viento"].astype("string").str.lower().str.strip().replace(sectores).str.upper()
        )

    estaciones = {
        "primavera": "spring", "verano": "summer", "otoño": "autumn", "otono": "autumn",
        "invierno": "winter", "primav": "spring", "primaveraa": "spring", "berano": "summer",
        "fall": "autumn", "invernio": "winter", "verno": "summer",
    }
    if "estacion_anio" in data:
        data["estacion_anio"] = data["estacion_anio"].astype("string").str.lower().str.strip().replace(estaciones)
        data.loc[~data["estacion_anio"].isin(["spring", "summer", "autumn", "winter"]), "estacion_anio"] = np.nan
        data["estacion_anio"] = data["estacion_anio"].replace(
            {"spring": "primavera", "summer": "verano", "autumn": "otono", "winter": "invierno"}
        )

    # Reconstrucción y consistencia de variables de fecha.
    if "fecha" in data:
        data["fecha"] = pd.to_datetime(data["fecha"], errors="coerce")
        if {"anio", "dia_del_anio"}.issubset(data.columns):
            faltante = data["fecha"].isna() & data["anio"].notna() & data["dia_del_anio"].notna()
            data.loc[faltante, "fecha"] = [
                obtener_fecha(anio, dia) for anio, dia in zip(data.loc[faltante, "anio"], data.loc[faltante, "dia_del_anio"])
            ]
        data["fecha"] = data["fecha"].ffill().bfill()
        meses_por_numero = {
            1: "January", 2: "February", 3: "March", 4: "April", 5: "May", 6: "June",
            7: "July", 8: "August", 9: "September", 10: "October", 11: "November", 12: "December",
        }
        estaciones_por_mes = {
            12: "invierno", 1: "invierno", 2: "invierno", 3: "primavera", 4: "primavera",
            5: "primavera", 6: "verano", 7: "verano", 8: "verano", 9: "otono", 10: "otono", 11: "otono",
        }
        mask = data["fecha"].notna()
        data.loc[mask, "anio"] = data.loc[mask, "fecha"].dt.year
        data.loc[mask, "dia_del_anio"] = data.loc[mask, "fecha"].dt.dayofyear
        data.loc[mask, "mes"] = data.loc[mask, "fecha"].dt.month.map(meses_por_numero)
        data.loc[mask, "estacion_anio"] = data.loc[mask, "fecha"].dt.month.map(estaciones_por_mes)

    if {"sector_viento", "direccion_viento"}.issubset(data.columns):
        faltante = data["sector_viento"].isna() & data["direccion_viento"].notna()
        data.loc[faltante, "sector_viento"] = data.loc[faltante, "direccion_viento"].apply(obtener_sector_compass)

    # Corrección de errores cuantitativos identificados en el Laboratorio 1.
    if "presion_media" in data:
        mask = data["presion_media"] > 9000
        data.loc[mask, "presion_media"] = data.loc[mask, "presion_media"].astype(str).str[1:].astype(float)
    for columna in ["viento_desv", "rafaga_desv"]:
        if columna in data:
            mask = data[columna] > 30
            data.loc[mask, columna] = data.loc[mask, columna].astype(str).str[1:].astype(float)
            data.loc[data[columna] > 30, columna] = np.nan
    if "viento_norte" in data:
        data.loc[data["viento_norte"] < -100, "viento_norte"] = np.nan
    for maximo, minimo in [("presion_max", "presion_min"), ("humedad_max", "humedad_min"),
                           ("viento_max", "viento_min"), ("rafaga_max", "rafaga_min")]:
        if {maximo, minimo}.issubset(data.columns):
            mask = data[maximo] < data[minimo]
            data.loc[mask, [maximo, minimo]] = data.loc[mask, [minimo, maximo]].to_numpy()
    humedad = [c for c in ["humedad_media", "humedad_min", "humedad_max"] if c in data]
    if humedad:
        data[humedad] = data[humedad].clip(lower=0, upper=100)
    if "rafaga_desv" in data:
        data.loc[data["rafaga_desv"] < -3, "rafaga_desv"] = np.nan
        mask = data["rafaga_desv"].between(-3, 0, inclusive="both")
        data.loc[mask, "rafaga_desv"] = data.loc[mask, "rafaga_desv"].abs()
    for columna in ["rafaga_media", "rafaga_min"]:
        if columna in data:
            data.loc[data[columna] < 0, columna] = np.nan

    # Se conserva un registro por observación meteorológica.
    data = data.drop_duplicates().reset_index(drop=True)
    for columna in ["mes", "estacion_anio", "sector_viento"]:
        if columna in data:
            data[columna] = data[columna].astype("category")
    return data

In [ ]:
data_train_raw = pd.read_csv(f"{DATA_DIR}/Datos Lab 1.csv")
data_test_raw = pd.read_csv(f"{DATA_DIR}/Datos Test Lab 1.csv")
data_train = limpiar_datos(data_train_raw)
data_test = limpiar_datos(data_test_raw)

print(f"Entrenamiento: {data_train.shape}; prueba externa: {data_test.shape}")
print("Faltantes del objetivo:", data_train[TARGET].isna().sum())

### 2.1 Variables derivadas, objetivo y partición

Las variables cíclicas representan la estacionalidad del día del año. El conjunto de entrenamiento (`Datos Lab 1.csv`) y el de prueba externa (`Datos Test Lab 1.csv`) se leyeron y limpiaron de forma totalmente independiente en la sección anterior, así que no comparten ninguna observación ni estadístico calculado entre sí. No se reserva un tercer conjunto de validación por separado: los hiperparámetros se buscan con validación cruzada (K-Fold) dentro del propio conjunto de entrenamiento, y el conjunto de prueba externa se deja intacto hasta la evaluación final del modelo seleccionado. De esta manera cada observación cumple un único rol (ajuste+validación cruzada, o evaluación final) y se evita el data leakage.

In [ ]:
def agregar_variables_ciclicas(df):
    resultado = df.copy()
    if "dia_del_anio" in resultado:
        resultado["dia_del_anio_sin"] = np.sin(2 * np.pi * resultado["dia_del_anio"] / 365.25)
        resultado["dia_del_anio_cos"] = np.cos(2 * np.pi * resultado["dia_del_anio"] / 365.25)
    return resultado


data_train = agregar_variables_ciclicas(data_train)
data_test = agregar_variables_ciclicas(data_test)
data_train = data_train.dropna(subset=[TARGET])
X_train = data_train.drop(columns=[TARGET])
y_train = data_train[TARGET]
X_test = data_test.drop(columns=[TARGET], errors="ignore")
y_test = data_test[TARGET] if TARGET in data_test else None

print(f"Train: {X_train.shape}; prueba externa: {X_test.shape}")

### 2.2 Pipeline de preprocesamiento

Las columnas `fecha`, `registros_del_dia` y `anio` no se usan como predictores directos. La imputación (mediana/moda) y el escalamiento se calculan dentro del pipeline, por lo que en cada fold de la validación cruzada se ajustan únicamente con el subconjunto de entrenamiento de ese fold, y no con los datos que se usan para medir el desempeño (ni con el conjunto de prueba externa). Esto es lo que evita la fuga de información durante la búsqueda de hiperparámetros.

In [ ]:
COLUMNAS_EXCLUIDAS = ["fecha", "registros_del_dia", "anio"]
numeric_features = [
    c for c in X_train.columns
    if c not in COLUMNAS_EXCLUIDAS and pd.api.types.is_numeric_dtype(X_train[c])
]
categorical_features = [
    c for c in X_train.columns
    if c not in COLUMNAS_EXCLUIDAS and c not in numeric_features
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="if_binary")),
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
], remainder="drop")

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
}
print("Variables numéricas:", len(numeric_features))
print("Variables categóricas:", categorical_features)

## 3. Actividad 1: regresión polinomial

Construir un pipeline `PolynomialFeatures + LinearRegression` y buscar `degree` y el escalamiento con `GridSearchCV`. Reportar RMSE, MAE y R², además de discutir el aumento de complejidad y el posible sobreajuste.

In [ ]:
# Completar y ejecutar la búsqueda cuando se decida el rango de grados.
polynomial_pipeline = Pipeline([
    ("preprocesamiento", preprocessor),
    ("polynomial", PolynomialFeatures(include_bias=False)),
    ("model", LinearRegression()),
])
polynomial_param_grid = {
    "polynomial__degree": [1, 2, 3],
    # "preprocesamiento__num__scaler": [StandardScaler(), RobustScaler()],
}
# polynomial_search = GridSearchCV(polynomial_pipeline, polynomial_param_grid, cv=cv,
#                                  scoring=scoring, refit="rmse", n_jobs=-1,
#                                  return_train_score=True)
# polynomial_search.fit(X_train, y_train)

## 4. Actividad 2: curvas de validación

Graficar el error promedio y su variabilidad para cada grado del polinomio. Interpretar el punto donde el error de entrenamiento sigue disminuyendo, pero el error de validación comienza a aumentar.

In [ ]:
# train_scores, valid_scores = validation_curve(
#     polynomial_pipeline, X_train, y_train,
#     param_name="polynomial__degree", param_range=[1, 2, 3, 4],
#     cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1,
# )
# degrees = [1, 2, 3, 4]
# plt.errorbar(degrees, -train_scores.mean(axis=1), yerr=train_scores.std(axis=1), label="Entrenamiento")
# plt.errorbar(degrees, -valid_scores.mean(axis=1), yerr=valid_scores.std(axis=1), label="Validación")
# plt.xlabel("Grado del polinomio"); plt.ylabel("RMSE"); plt.legend(); plt.show()

## 5. Actividad 3: penalización L1/L2 sobre el modelo lineal

Ridge (L2) y Lasso (L1) son penalizaciones sobre la magnitud de los coeficientes, no un tipo de modelo aparte: pueden aplicarse tanto sobre las variables originales (caso lineal, aquí) como sobre features polinomiales (Actividad 4). Comparar el modelo lineal sin regularización con Ridge y Lasso sobre las variables originales. Reportar el efecto de `alpha` sobre las métricas, la magnitud de los coeficientes y la selección automática de variables de Lasso.

In [ ]:
linear_models = {
    "lineal": LinearRegression(),
    "ridge": Ridge(),
    "lasso": Lasso(max_iter=20000, random_state=RANDOM_STATE),
}
regularized_searches = {}
for name, model in linear_models.items():
    pipeline = Pipeline([("preprocesamiento", preprocessor), ("model", model)])
    params = {} if name == "lineal" else {"model__alpha": np.logspace(-3, 2, 6)}
    # regularized_searches[name] = GridSearchCV(
    #     pipeline, params, cv=cv, scoring=scoring, refit="rmse", n_jobs=-1,
    #     return_train_score=True,
    # )
    # regularized_searches[name].fit(X_train, y_train)

## 6. Actividad 4: penalización L1/L2 sobre features polinomiales

Se repite la penalización de Ridge/Lasso de la Actividad 3, pero ahora sobre `PolynomialFeatures`: la regularización no es exclusiva del modelo lineal simple, sino que se combina igual de bien con features no lineales. Explorar simultáneamente el grado, `alpha` y la estrategia de escalamiento. Discutir si la penalización controla el sobreajuste al incrementar la complejidad.

In [ ]:
polynomial_regularized_pipeline = Pipeline([
    ("preprocesamiento", preprocessor),
    ("polynomial", PolynomialFeatures(include_bias=False)),
    ("model", Ridge()),
])
polynomial_regularized_param_grid = {
    "polynomial__degree": [1, 2, 3],
    "model__alpha": np.logspace(-2, 2, 5),
}
# polynomial_regularized_search = GridSearchCV(
#     polynomial_regularized_pipeline, polynomial_regularized_param_grid,
#     cv=cv, scoring=scoring, refit="rmse", n_jobs=-1,
# )
# polynomial_regularized_search.fit(X_train, y_train)

## 7. Actividad 5: comparación y selección del mejor modelo

Construir una tabla con RMSE, MAE, R², desviaciones estándar de validación cruzada, hiperparámetros y complejidad. Seleccionar el modelo considerando estabilidad, no solo la métrica promedio.

In [ ]:
resultados_modelos = pd.DataFrame(columns=[
    "modelo", "rmse_cv", "rmse_std", "mae_cv", "r2_cv", "hiperparametros", "complejidad"
])
resultados_modelos

## 8. Actividad 6: evaluación final e intervalos de confianza

Ajustar el modelo seleccionado con todo el entrenamiento, evaluar el conjunto de prueba y aplicar al menos 500 remuestreos bootstrap sobre el conjunto de test. Reportar los intervalos de confianza al 95% para RMSE, MAE y R², y graficar sus distribuciones.

In [ ]:
def calcular_metricas(y_real, y_pred):
    return {
        "RMSE": mean_squared_error(y_real, y_pred, squared=False),
        "MAE": mean_absolute_error(y_real, y_pred),
        "R2": r2_score(y_real, y_pred),
    }


# modelo_final = ...
# modelo_final.fit(X_train, y_train)
# predicciones_test = modelo_final.predict(X_test)
# metricas_test = calcular_metricas(y_test, predicciones_test)

# rng = np.random.default_rng(RANDOM_STATE)
# bootstrap_metricas = []
# for _ in range(500):
#     indices = rng.integers(0, len(y_test), len(y_test))
#     bootstrap_metricas.append(calcular_metricas(y_test.iloc[indices], predicciones_test[indices]))
# bootstrap_metricas = pd.DataFrame(bootstrap_metricas)
# intervalos = bootstrap_metricas.quantile([0.025, 0.975]).T.rename(columns={0.025: "limite_inferior", 0.975: "limite_superior"})
# intervalos

## 9. Análisis de resultados

### 9.1 Análisis cuantitativo

- ¿Cuál modelo obtuvo el mejor desempeño en test?
- ¿Coincide con el mejor promedio de validación cruzada? Si no, ¿por qué?
- ¿Cómo cambia el error con la complejidad y dónde aparece sobreajuste?
- ¿Cómo afecta la regularización la magnitud y estabilidad de los coeficientes?
- ¿Qué indican los intervalos bootstrap sobre la estabilidad?

### 9.2 Análisis cualitativo y conceptual

- ¿Qué variables selecciona Lasso y qué interpretación práctica tienen?
- ¿Qué diferencias existen entre precisión e interpretabilidad?
- ¿Qué relación hay entre complejidad, generalización y estabilidad?
- ¿Qué sesgos pueden estar presentes y cómo cambiaría el análisis con más datos?

**Respuestas del equipo:**

_Completar después de ejecutar y verificar los resultados._

## 10. Comunicación y entregables

- Preparar el video de máximo tres minutos dirigido al ingeniero de machine learning de AlpesPlanck.
- Exportar este notebook a `.html` con las ejecuciones visibles.
- Incluir las justificaciones metodológicas, respuestas del análisis y tabla comparativa.

## 11. Uso de herramientas de IA generativa

Documentar el nombre de la herramienta, los prompts principales, el análisis crítico de sus respuestas y los aportes propios del equipo. Explicar qué decisiones fueron verificadas o modificadas y qué aprendizajes se obtuvieron.